[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ersilia-os/ub-cedd-projects-workshop/blob/main/projects/blue/notebooks/blue_chemical_space.ipynb)

# Exploring the chemical space of the CpABC1 pharmacophore hits

**Blue group · Cryptosporidiosis**

Silymarin slows *Cryptosporidium parvum* down, but only at concentrations far too high to
be a medicine. The group turned the CpABC1–silymarin complex into a pharmacophore and
screened a purchasable library with it, which returned 28,732 hits. This notebook maps
those hits and asks the question that decides what to do next: are they relatives of
silymarin, or has the pharmacophore found something new?

## What you will do

- Load the 28,732 Pharmit hits together with their chemical space coordinates from `eos1klk`
- Describe the hits with simple molecular properties, and compare them with silymarin
- Draw the chemical space four different ways, and mark silymarin on it
- Rank the hits by how much they look like silymarin, and pick a short, varied list to take forward

## Setup

Run the cell below first. In Colab it downloads the workshop repository (including the data) and installs the packages this project needs. It takes about a minute. **Don't change it.**

In [ ]:
PROJECT = "blue"
NEEDS_GPU = False
import os, sys, shutil, subprocess
if "google.colab" in sys.modules:
    repo_dir = "/content/ub-cedd-projects-workshop"
    if not os.path.exists(repo_dir):
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/ersilia-os/ub-cedd-projects-workshop.git", repo_dir], check=True)
    else:
        subprocess.run(["git", "-C", repo_dir, "pull", "--ff-only"], check=True)
    os.chdir(f"{repo_dir}/projects/{PROJECT}")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
sys.path.insert(0, os.getcwd())
for _cached in [m for m in sys.modules if m == "scripts" or m.startswith("scripts.")]:
    del sys.modules[_cached]  # forget helper modules imported before the pull above
has_gpu = shutil.which("nvidia-smi") is not None and subprocess.run(["nvidia-smi"], capture_output=True).returncode == 0
print(f"Python {sys.version.split()[0]} | GPU: {'yes' if has_gpu else 'no'} | Folder: {os.getcwd()}")
if NEEDS_GPU and not has_gpu:
    print("WARNING: this notebook needs a GPU. Go to Runtime > Change runtime type, choose CPU, and run this cell again.")

## 1. The hits and the seed molecule

Three files go into this notebook.

`data/pharmit_hits_molport.csv` is the hit list: one row per molecule, with its SMILES
and its MolPort catalogue number. Every one of them can be bought, which is what makes
this list worth screening.

`data/eos1klk_pharmit_hits.csv` is the output of the Ersilia model
[eos1klk](https://github.com/ersilia-os/eos1klk). The model takes a molecule and returns
eight numbers: a pair of coordinates on each of four different maps. The maps were built
from the Ersilia Reference Library, 1.3 million compounds, so a molecule's position says
where it sits among molecules in general, not only among these hits.

`data/eos1klk_silymarin.csv` is the same model run on one molecule, silymarin, so that
the seed can be drawn on the same maps as the hits.

> **Note:** the hits have no measured activity. Nothing here has been
> tested against the parasite, so there are no actives and inactives to colour the map
> by. The one molecule that *is* known to work is silymarin, and that is why it does so
> much of the work in this notebook.

Load the hit list and join it to the coordinates, molecule by molecule, on the SMILES.

In [ ]:
import pandas as pd
import stylia
from scripts import chemspace

space = chemspace.load_space("data/pharmit_hits_molport.csv", "data/eos1klk_pharmit_hits.csv")
print(f"{len(space):,} purchasable hits, each with coordinates on four maps")
space.head()

These are the eight coordinate columns. Each pair is one map: `pca_x` and
`pca_y` place the molecule on the PCA map, `umap_x` and `umap_y` on the UMAP map, and so
on. The numbers have no units and mean nothing on their own; only the distance between
two molecules does.

In [ ]:
space.filter(regex="_(x|y)$").head()

Now the seed. Silymarin is written here without stereochemistry: the model
that made the maps does not read stereochemistry, and the structure the group docked had
its stereocentres guessed from the pose rather than measured.

In [ ]:
SILYMARIN = "COc1cc(C2Oc3cc(C4Oc5cc(O)cc(O)c5C(=O)C4O)ccc3OC2CO)ccc1O"

seed = pd.read_csv("data/eos1klk_silymarin.csv").drop(columns=["key"], errors="ignore")
seed = seed.rename(columns={"input": "smiles"})
chemspace.check_seed(seed, SILYMARIN)  # the coordinates must belong to this molecule
chemspace.draw_molecules([SILYMARIN], ["silymarin (the seed)"], per_row=1)

## 2. What kind of molecules are these?

With no activity to plot, the first thing to look at is the molecules themselves. A
handful of simple properties — how heavy a molecule is, how greasy, how many hydrogen
bonds it can make — say a lot about whether something could ever become a drug, and they
are quick to work out from the structure alone.

The question to keep in mind is whether the hits resemble silymarin on these properties.
A pharmacophore describes a *pattern* of chemical features in space, not a shape, so it
can perfectly well return molecules that share silymarin's binding features while looking
nothing like it.

Set the plotting style once. Every plot in this notebook uses `stylia`, so
they all come out with the same fonts and colours.

In [ ]:
stylia.set_format("slide")
stylia.set_style("ersilia")
nc = stylia.NamedColors()  # nc.blue, nc.gray, nc.plum, nc.pink, nc.mint ...

print("Plots in this notebook use the Ersilia style, in the blue group's colour.")

Work out the six properties for every hit, and for silymarin. This takes
about fifteen seconds, because each of the 28,732 SMILES has to be turned into a molecule
first.

In [ ]:
properties = chemspace.describe(space["smiles"])
space[properties.columns] = properties
silymarin_properties = chemspace.describe([SILYMARIN]).iloc[0]

pd.DataFrame({"hits (median)": properties.median(), "silymarin": silymarin_properties}).round(1)

A histogram cuts a property's range into bins and counts how many molecules
fall in each one. The dashed line is where silymarin sits, so each panel shows at a glance
whether the seed is typical of the hits or off to one side.

In [ ]:
fig, axs = stylia.create_figure(2, 3, width=1.0, height=0.7)
for key, (name, _) in chemspace.DESCRIPTORS.items():
    ax = axs.next()
    ax.hist(space[key], bins=40, color=nc.blue)
    ax.axvline(silymarin_properties[key], color=nc.plum, linestyle="--")
    stylia.label(ax, xlabel=name, ylabel="Hits")

> **Exercise:** which of the six panels has silymarin far from the bulk of
> the hits, and which has it in the middle? A property where the seed is an outlier is
> one the pharmacophore did not constrain, and it is worth asking whether that matters
> for getting into the parasite.

## 3. The map of chemical space

A molecule has thousands of properties, so it cannot be drawn on a page as it is. A
projection squeezes all of them into two numbers, keeping molecules that are alike close
together. `eos1klk` gives four of them, and they disagree on purpose:

- **PCA** is the plainest: it keeps the big distances honest, so far-apart points really
  are different, but everything piles up in the middle.
- **t-SNE** pulls neighbours together and separates groups clearly. The distance between
  two separate clusters means nothing.
- **UMAP** does something similar but keeps a little more of the overall shape.
- **TMAP** lays the molecules out as a tree, which spreads out sparse regions.

None of them has a right answer. Read them together: a group of molecules that stays
together in all four is a real family.

Start with UMAP, one point per hit.

In [ ]:
fig, axs = stylia.create_figure(1, 1, width=0.5, height=0.5)
ax = axs.next()
chemspace.plot_points(ax, space, "umap", color=nc.blue, alpha=0.2, size=4)
chemspace.label_space(ax, "umap", f"{len(space):,} Pharmit hits")

Now the same hits on all four maps, side by side.

In [ ]:
fig, axs = stylia.create_figure(2, 2, width=1.0, height=1.0)  # a square figure
for projection, letter in zip(chemspace.PROJECTIONS, "ABCD"):
    ax = axs.next()
    chemspace.plot_points(ax, space, projection, color=nc.blue, alpha=0.2, size=4)
    ax.set_box_aspect(1)  # each panel is a square, whatever range its numbers cover
    chemspace.label_space(ax, projection, chemspace.PROJECTIONS[projection], abc=letter)

> **Exercise:** pick a map and describe what you see. How many separate
> clumps are there? Does the same number of clumps show up in the other three maps? The
> rest of this notebook uses UMAP, but every cell below works with `"pca"`, `"tsne"` or
> `"tmap"` instead.

## 4. Where does silymarin sit?

This is the section the whole project turns on. The pharmacophore was built from
silymarin bound to CpABC1, so one might expect the hits to be silymarin-like. Whether
they are is an empirical question, and the map answers it: draw the hits as a faint
backdrop and put the seed on top as a star.

If the star lands inside the crowd, the pharmacophore has found more of the same
chemistry, and silymarin's known activity is decent evidence that the hits are worth
testing. If it lands outside, the hits satisfy the same arrangement of features using
different chemistry, which is more interesting and more of a gamble.

Silymarin on the UMAP map, with the hits behind it.

In [ ]:
fig, axs = stylia.create_figure(1, 1, width=0.5, height=0.5)
ax = axs.next()
chemspace.plot_backdrop(ax, space, "umap")
chemspace.plot_seed(ax, seed, "umap", color=nc.plum)
ax.legend()
chemspace.label_space(ax, "umap", "Silymarin among its own hits")

And on all four maps, in case one of them disagrees.

In [ ]:
fig, axs = stylia.create_figure(2, 2, width=1.0, height=1.0)
for projection, letter in zip(chemspace.PROJECTIONS, "ABCD"):
    ax = axs.next()
    chemspace.plot_backdrop(ax, space, projection)
    chemspace.plot_seed(ax, seed, projection, color=nc.plum)
    ax.set_box_aspect(1)
    chemspace.label_space(ax, projection, chemspace.PROJECTIONS[projection], abc=letter)

> **Exercise:** write down, in one sentence, whether silymarin sits inside
> the hits or outside them, and whether the four maps agree. Then say what that means for
> the group: does the hit list need a silymarin-like molecule in it as a positive control,
> or does it already have plenty?

## 5. How much do the hits look like silymarin?

The map is a summary, and summaries lose detail: two molecules can land on the same spot
without being related. To measure resemblance directly, compare **fingerprints**.

A fingerprint describes a molecule as a long list of yes/no answers, one for each small
fragment that might be present. The **Tanimoto similarity** between two fingerprints runs
from 0, nothing in common, to 1, the same fragments throughout. Above roughly 0.4 two
molecules are usually recognisable as relatives; above 0.7 they are close analogues.

Fingerprint every hit and score it against silymarin. This takes a few seconds.

In [ ]:
prints = chemspace.fingerprints(space["smiles"])
silymarin_print = chemspace.fingerprints([SILYMARIN])[0]
space["similarity"] = chemspace.tanimoto(prints, silymarin_print)

print(f"most similar hit: {space['similarity'].max():.2f} | "
      f"median: {space['similarity'].median():.2f} | "
      f"hits above 0.4: {(space['similarity'] > 0.4).sum()}")

The whole distribution, with the 0.4 mark drawn in.

In [ ]:
fig, axs = stylia.create_figure(1, 1)
ax = axs.next()
ax.hist(space["similarity"], bins=50, color=nc.blue)
ax.axvline(0.4, color=nc.plum, linestyle="--")
stylia.label(ax, xlabel="Tanimoto similarity to silymarin", ylabel="Hits",
             title="How closely the hits resemble the seed")

Now put that on the map. Colouring all 28,732 points by similarity does not
work here, because almost every one of them scores low and the whole map comes out the
same shade. Highlighting the 200 most similar instead asks the question directly: do the
hits that resemble silymarin gather near it, or are they scattered everywhere?

In [ ]:
closest_200 = space.nlargest(200, "similarity")

fig, axs = stylia.create_figure(1, 1, width=0.5, height=0.5)
ax = axs.next()
chemspace.plot_backdrop(ax, space, "umap")
chemspace.plot_points(ax, closest_200, "umap", color=nc.blue, alpha=0.9, size=16,
                      label=f"200 most similar ({closest_200['similarity'].min():.2f}–"
                            f"{closest_200['similarity'].max():.2f})")
chemspace.plot_seed(ax, seed, "umap", color=nc.plum)
ax.legend()
chemspace.label_space(ax, "umap", "The hits that most resemble silymarin")

These are the eight hits that resemble silymarin most, with their catalogue numbers.

In [ ]:
closest = space.nlargest(8, "similarity")
chemspace.draw_molecules(closest["smiles"],
                         [f"{row.molport_id}\n{row.similarity:.2f}" for row in closest.itertuples()])

> **Exercise:** look at the eight molecules above next to the drawing of
> silymarin in section 1. What have they kept from it, and what have they dropped? A hit
> that keeps the same features but is smaller and less floppy is exactly what the project
> is after, because silymarin's size is one reason it needs such high concentrations.

## 6. Scaffolds, and a shortlist to take forward

28,732 molecules is too many to dock, and far too many to buy. The last step is cutting
the list down, and there are two opposite ways to do it.

The first is to group the hits by **scaffold** — what is left of a molecule once every
side chain is removed, so its rings and the links between them. Molecules that share a
scaffold are variations on one idea, and counting scaffolds shows how many genuinely
different ideas the hit list contains.

Work out the scaffold of every hit and count the most common ones.

In [ ]:
space["scaffold"] = chemspace.murcko_scaffolds(space["smiles"])
common = space["scaffold"].value_counts()
print(f"{space['scaffold'].nunique():,} different scaffolds among {len(space):,} hits | "
      f"{space['scaffold'].isna().sum()} molecules have no rings at all")
common.head(6).rename("hits")

This is what those six look like, numbered from the most common one down.

In [ ]:
top = common.head(6)
chemspace.draw_molecules(top.index, [f"{i}: {n} hits" for i, n in enumerate(top.values, 1)],
                         per_row=3)

Put them on the map. Each scaffold keeps the number it has in the drawing
above, and everything else stays grey.

In [ ]:
palette = stylia.CategoricalPalette("ersilia").get(len(top))

fig, axs = stylia.create_figure(1, 1, width=0.5, height=0.5)
ax = axs.next()
chemspace.plot_backdrop(ax, space, "umap")
for rank, ((scaffold, size), color) in enumerate(zip(top.items(), palette), 1):
    chemspace.plot_points(ax, space[space["scaffold"] == scaffold], "umap",
                          color=color, label=f"{rank} ({size})", alpha=0.9, size=12)
ax.legend()
chemspace.label_space(ax, "umap", "The six most common scaffolds")

The other way to cut the list is the opposite of picking the most similar
molecules: pick the ones least like *each other*. The MaxMin algorithm starts from one
molecule and repeatedly adds whichever is furthest from everything chosen so far, so
twelve picks cover the whole range of chemistry rather than twelve versions of one idea.
This is the right shortlist when nothing has been tested yet and the aim is to learn as
much as possible from the first round.

There is a catch. "Furthest from everything else" is exactly what an odd molecule is, so
run MaxMin on the raw hit list and it hands back the twelve strangest things in it —
molecules of 1,400 daltons, or with a logP of 17, which no laboratory would order. Filter
the list to sensible molecules first, and MaxMin then spreads its picks across chemistry
that is actually worth buying.

Keep only the hits inside the usual drug-like range: not too heavy, not too
greasy, not too many hydrogen bonds. Silymarin itself passes these limits, so the filter
is not quietly throwing away the kind of molecule the group is looking for.

In [ ]:
druglike = space[(space["mw"] <= 500) & space["logp"].between(0, 5)
                 & (space["hbd"] <= 5) & (space["hba"] <= 10)].reset_index(drop=True)

print(f"{len(druglike):,} of {len(space):,} hits are drug-like ({len(druglike) / len(space):.0%})")
pd.DataFrame({"drug-like hits (median)": chemspace.describe(druglike["smiles"]).median(),
              "silymarin": silymarin_properties}).round(1)

Now pick twelve of those that are as unlike each other as possible.

In [ ]:
picked = druglike.iloc[chemspace.diverse_subset(chemspace.fingerprints(druglike["smiles"]), 12)]
print(f"similarity to silymarin in this set: "
      f"{picked['similarity'].min():.2f} to {picked['similarity'].max():.2f}")
picked[["molport_id", "similarity", "mw", "logp"]].round(2)

Where those twelve land on the map, against the rest of the hits.

In [ ]:
fig, axs = stylia.create_figure(1, 1, width=0.5, height=0.5)
ax = axs.next()
chemspace.plot_backdrop(ax, space, "umap")
chemspace.plot_points(ax, picked, "umap", color=nc.plum, label="the twelve picked", alpha=0.9, size=60)
chemspace.plot_seed(ax, seed, "umap", color=nc.pink)
ax.legend()
chemspace.label_space(ax, "umap", "A shortlist that covers the whole map")

> **Exercise:** the two shortlists in this notebook answer different
> questions. The eight most similar hits ask *can we do better than silymarin at what
> silymarin already does*; the twelve most varied ask *what else satisfies this
> pharmacophore*. Build a third list of twenty that mixes them, decide the split, and
> write down why. That list is what goes to docking.

> **Note:** to put any other molecules on these maps — a new Pharmit run, a
> set of known ABC transporter inhibitors, whatever the group finds — write their SMILES
> into a one-column file, run it through `eos1klk` in Ersilia, upload the output to
> **Projects/BlueTeam/Data** as `eos1klk_<what the molecules are>.csv`, and ask for it to
> be copied into `data/` here. Nothing in this notebook needs a GPU or an internet
> connection once the file exists.

## Summary

- You mapped 28,732 purchasable Pharmit hits with coordinates from the Ersilia model
  `eos1klk`, and put silymarin, the molecule the pharmacophore was built from, on the
  same maps.
- On simple properties the hits are lighter and much less polar than silymarin: a median
  of 412 daltons against 482, and a polar surface area of 84 against 155. That is the
  direction the project wants, because silymarin's size and its many hydroxyl groups are
  part of why it only works at 100 micromolar.
- Silymarin sits at the **edge** of its own hit list, not in the middle of it, and all
  four maps agree. On UMAP it is further out than 93% of the hits along one axis, and
  only about 8% of them are anywhere near it.
- Fingerprint similarity to silymarin is low everywhere: the closest hit scores 0.46 and
  the median is 0.11, with only two hits above 0.4. The pharmacophore has reproduced
  silymarin's binding features using quite different chemistry, which is worth knowing
  before anything is bought. The 200 hits that do resemble it are concentrated in the
  upper part of the map, near where silymarin itself lands.
- The hit list is genuinely varied, with 20,900 scaffolds among 28,732 molecules, so no
  single series dominates it.
- Two shortlists came out of the same data: the hits most like silymarin, and a spread of
  twelve drug-like hits that covers the whole map.

**Next:** dock the shortlist against the AlphaFold model of CpABC1 and compare the scores
with silymarin's, then use the ADMET models in the Ersilia Model Hub to drop anything that
could never become a medicine.